# Structured output i stan agenta

Klient pisze wiadomość po swojemu. Chcemy dostać z niej **trzy konkretne pola**, których może użyć kod Pythona. Zobaczymy też, gdzie agent trzyma te dane po zakończeniu pracy.

W tej lekcji jest jedno zgłoszenie, jeden schemat i jedno wywołanie agenta. Niczego nie zapisujemy ani nie wysyłamy do systemu płatności.

In [1]:
# Wszystkie importy i minimalne ustawienie ścieżki projektu są w tej komórce.
import sys
from pathlib import Path
from typing import Literal

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from model_config import get_model
from pydantic import BaseModel

## 1. Wiadomość i schemat

`SupportCase` opisuje wynik, którego oczekujemy:

- `order_id` — numer zamówienia;
- `issue` — jedna z dwóch dozwolonych kategorii;
- `summary` — krótkie podsumowanie po polsku.

`Literal` ogranicza możliwe wartości `issue`. Pydantic sprawdzi typy i dozwoloną kategorię. Nie sprawdzi, czy model dobrze zrozumiał wiadomość.

In [2]:
SOURCE_TEXT = (
    "Dzień dobry. Zamówienie A-174. Kartą pobrano 79 zł dwa razy, "
    "chociaż kupiłam tylko jeden bilet. Proszę sprawdzić płatność."
)

class SupportCase(BaseModel):
    order_id: str
    issue: Literal["double_charge", "other"]
    summary: str

print(SOURCE_TEXT)

Dzień dobry. Zamówienie A-174. Kartą pobrano 79 zł dwa razy, chociaż kupiłam tylko jeden bilet. Proszę sprawdzić płatność.


## 2. Agent ma zwrócić taki obiekt

`response_format=ToolStrategy(SupportCase)` przekazuje agentowi schemat. Agent użyje mechanizmu wywołań narzędzi do oddania danych w tym formacie. Nie mamy tu żadnego narzędzia wykonującego operacje biznesowe (`tools=[]`).

Model pochodzi ze wspólnej konfiguracji projektu.

In [3]:
model = get_model()
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(SupportCase),
    system_prompt=(
        "Wyodrębnij dane zgłoszenia. Nie dopisuj faktów. "
        "Podsumowanie napisz po polsku."
    ),
)

MODEL [main] profile=local provider=ollama model=gemma4:12b


## 3. Wejście i stan agenta

Na wejściu stan zawiera `messages` z wiadomością klienta. `agent.invoke(...)` zwróci **stan po zakończeniu przebiegu**. Będą w nim komunikaty z przebiegu oraz `structured_response`, czyli obiekt zgodny z naszym schematem.

In [4]:
input_state = {"messages": [{"role": "user", "content": SOURCE_TEXT}]}
print("Stan na wejściu:", input_state)

result = agent.invoke(input_state)

Stan na wejściu: {'messages': [{'role': 'user', 'content': 'Dzień dobry. Zamówienie A-174. Kartą pobrano 79 zł dwa razy, chociaż kupiłam tylko jeden bilet. Proszę sprawdzić płatność.'}]}


## 4. Co agent zwrócił?

`messages` pokazuje przebieg: wiadomość człowieka oraz komunikaty związane z odpowiedzią modelu. `structured_response` to wynik, który najłatwiej wykorzystać w Pythonie. Przy `ToolStrategy` ostatni komunikat nie musi być zwykłą odpowiedzią tekstową, więc czytamy właśnie `structured_response`.

In [6]:
print("Klucze stanu:", list(result))
print("Typy komunikatów:", [message.type for message in result["messages"]])


Klucze stanu: ['messages', 'structured_response']
Typy komunikatów: ['human', 'ai', 'tool']


In [11]:
case = result["structured_response"]
case


SupportCase(order_id='A-174', issue='double_charge', summary='Klientka zgłasza podwójne pobranie opłaty (79 zł) za jeden zakupiony bilet.')

In [ ]:
print("Dane dla Pythona:", case.model_dump())

Dane dla Pythona: {'order_id': 'A-174', 'issue': 'double_charge', 'summary': 'Klientka zgłasza podwójne pobranie opłaty (79 zł) za jeden zakupiony bilet.'}


In [8]:

queue = "płatności" if case.issue == "double_charge" else "inne zgłoszenia"
print("Proponowana kolejka:", queue)

Proponowana kolejka: płatności


## 5. Jedna odpowiedź, różne zastosowania pól

Teraz poprosimy model o inny ustrukturyzowany wynik:

- `text` — odpowiedź, którą może zobaczyć klient;
- `reasoning` — krótkie uzasadnienie dla aplikacji lub operatora;
- `should_call_it` — propozycja, czy płatność wymaga sprawdzenia w innym systemie.

`reasoning` to **wygenerowane uzasadnienie**, a nie podgląd ukrytego procesu myślenia modelu. Flaga jest decyzją modelu, którą kod może wykorzystać do wybrania następnego kroku.

In [ ]:
class ReplyPlan(BaseModel):
    text: str
    reasoning: str
    should_call_it: bool


reply_agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ReplyPlan),
    system_prompt=(
        "Odpowiedz klientowi po polsku. "
        "text: krótka odpowiedź dla klienta, bez twierdzenia, że płatność już sprawdzono. "
        "reasoning: jedno krótkie zdanie, dlaczego potrzebna jest weryfikacja albo nie. "
        "should_call_it: true, jeśli sprawę trzeba sprawdzić w systemie płatności."
    ),
)

reply_result = reply_agent.invoke({
    "messages": [{"role": "user", "content": SOURCE_TEXT}]
})
reply = reply_result["structured_response"]
reply

In [ ]:
print("Dla klienta:", reply.text)
print("Uzasadnienie dla aplikacji:", reply.reasoning)
print("Czy proponować weryfikację płatności?", reply.should_call_it)

if reply.should_call_it:
    next_step = "Wpisz zgłoszenie do kolejki weryfikacji płatności"
else:
    next_step = "Nie wpisuj zgłoszenia do kolejki weryfikacji płatności"
print("Symulowany następny krok:", next_step)

Kod wybrał napis na podstawie `should_call_it`. Nie wywołał systemu płatności ani nie zmienił zgłoszenia. `text` jest dla klienta, a `reasoning` i flaga pozostają po stronie aplikacji. Schemat pilnuje typów pól; poprawności decyzji modelu trzeba pilnować osobno.

[Dokumentacja structured output](https://docs.langchain.com/oss/python/langchain/structured-output)